# Kling V3 Omni Video-to-Video Edit API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Kling V3 Omni (Video-to-Video Edit)** 模型 API。

## 模型简介

Kling V3 Omni Video-to-Video Edit 以输入视频为**编辑基底**（`refer_type=base`）直接修改视频内容：

- **直接内容编辑**：模型在保持视频结构、时长、尺寸的基础上，按 prompt 修改视频内容
- **尺寸与时长固定**：输出沿用输入视频的原始尺寸和时长（与 reference 端点的关键区别）
- **场景替换**：可改变场景环境、天气、风格等，同时保留主体运动

**API 端点**：`fal-ai/kling-video/o3/{mode}/video-to-video/edit`

**Mode 选项**：`standard`（720P）、`pro`（1080P）

> **注意**：video-to-video 系列不支持 `4k` mode。

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 编辑视频场景

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：编辑视频场景（standard 模式）
# edit 端点不需要传 aspect_ratio 和 duration，输出沿用输入视频
result = run_fal_task(
    "fal-ai/kling-video/o3/standard/video-to-video/edit",
    arguments={
        "prompt": "Transform the forest scene in @Video1 into autumn with golden leaves falling while the man keeps running",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 描述如何编辑视频内容，可用 `@Video1` 引用输入视频 |
| `video_url` | string | *必填* | 待编辑的输入视频 URL |
| `image_urls` | list[string] | 可选 | 额外图片参考 URL 列表 |
| `keep_audio` | boolean | `true` | 是否保留输入视频的原始音频 |
| `shot_type` | string | 可选 | 镜头类型，固定传 `"customize"` |

> **注意**：edit 端点不支持 `aspect_ratio` 和 `duration`，输出始终与输入视频保持相同尺寸和时长。

## 3. Pro 模式 — 更高分辨率输出

In [ ]:
# Pro 模式：1080P 输出
result_pro = run_fal_task(
    "fal-ai/kling-video/o3/pro/video-to-video/edit",
    arguments={
        "prompt": "Transform the forest scene in @Video1 into autumn with golden leaves falling while the man keeps running",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
    },
)

video_url = result_pro["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 高级用法 — 关闭原始音频

In [ ]:
# 关闭原始音频
result_no_audio = run_fal_task(
    "fal-ai/kling-video/o3/pro/video-to-video/edit",
    arguments={
        "prompt": "Transform the forest scene in @Video1 into autumn with golden leaves falling while the man keeps running",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
        "keep_audio": False,
    },
)

video_url = result_no_audio["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/kling-video/o3/standard/video-to-video/edit",
    arguments={
        "prompt": "Transform the forest scene in @Video1 into autumn with golden leaves falling while the man keeps running",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/kling-video/o3/standard/video-to-video/edit",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/kling-video/o3/standard/video-to-video/edit",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 6. Schema 参考

### Input Schema

```json
{
  "prompt": "string (必填)",
  "video_url": "string (必填, 待编辑的输入视频 URL)",
  "image_urls": ["string (可选, 额外图片参考 URL 列表)"],
  "keep_audio": "boolean (可选, 默认 true)",
  "shot_type": "string (可选, 固定传 customize)"
}
```

### Output Schema

```json
{
  "video": {
    "file_name": "string",
    "url": "string",
    "content_type": "string",
    "file_size": "number"
  }
}
```

### 与 video-to-video/reference 的区别

| 特性 | edit（本文档） | reference |
|------|--------------|----------|
| refer_type | `base`（直接编辑） | `feature`（特征参考） |
| aspect_ratio | 沿用输入视频尺寸 | 支持自定义 |
| duration | 沿用输入视频时长 | 支持自定义（3~15秒） |
| 适用场景 | 对视频内容直接修改 | 参考视频风格/动作生成新内容 |

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Kling V3 Omni 模型页面](https://fal.ai/models/fal-ai/kling-video/o3/standard/video-to-video/edit)